# Grid Outage Planner Evaluation Notebook

This notebook evaluates the Grid Outage Forecaster on the final 30 days of synthetic hourly history. It reports Brier score, duration MAE, lead time, outage-factor summaries, and the five worst forecast hours.

In [ ]:
from pathlib import Path
import pandas as pd

from generate_data import generate_all
from forecaster import load_history, train, forecast_next_24, evaluate_holdout

if not Path('data/grid_history.csv').exists():
    generate_all('data')

history = load_history('data/grid_history.csv')
metrics, worst_cases = evaluate_holdout(history)
pd.DataFrame([metrics])

## Metric Interpretation

- **Brier score**: probability forecast error for outage/no-outage. Lower is better.
- **Duration MAE**: average absolute error in expected duration on true outage hours.
- **Lead time**: how early the model raised an alert before held-out outage hours, using the documented threshold.

In [ ]:
cols = [c for c in ['timestamp', 'load_mw', 'rain_mm', 'voltage_drop_index', 'feeder_congestion_index', 'maintenance_flag', 'neighbor_outage_reports', 'primary_outage_driver', 'outage', 'duration_min', 'p_outage', 'expected_duration_min', 'probability_error', 'duration_abs_error'] if c in worst_cases.columns]
worst_cases[cols]

## Outage Factor Summary

The next cell confirms that outage risk is not based on appliances. Appliances are only used later for the action plan.

In [ ]:
factor_cols = [c for c in ['load_stress_index', 'rain_stress_index', 'wind_stress_index', 'feeder_congestion_index', 'voltage_drop_index', 'maintenance_flag', 'neighbor_outage_reports', 'transformer_age_years', 'payment_day_flag', 'reserve_margin_index', 'fuel_supply_risk_index', 'hydro_inflow_stress_index', 'vegetation_risk_index', 'protection_miscoordination_index', 'scada_telecom_risk_index', 'non_technical_loss_index', 'asset_health_index', 'der_backup_risk_index'] if c in history.columns]
summary = history[factor_cols].mean().to_frame('average_value')
driver_counts = history.loc[history['outage'] == 1, 'primary_outage_driver'].value_counts().to_frame('outage_hours') if 'primary_outage_driver' in history.columns else pd.DataFrame()
summary, driver_counts

## 24-Hour Forecast Preview

The next cell trains on all available synthetic history and previews the next 24-hour forecast table used by the prioritizer.

In [ ]:
model = train(history)
forecast = forecast_next_24(model, history)
forecast.head(24)

## Result Explanation For Reviewers

The forecast is useful as a planning signal when it highlights higher-risk windows early enough for the business to prepare appliances. The worst cases show where factor signals can create false alarms or miss events. The most useful next data would be real feeder logs, verified utility alerts, and neighbor reports.